# Ingredient Detection to Recipe Generation Adapter

**Purpose**: Convert Roboflow detection outputs to recipe generation model inputs

**Input**: Roboflow detection results (multiple bounding boxes, classes, confidences)

**Output**: Formatted prompts for GPT-2/Transformer recipe generation

**Example**:
```
Input:  {predictions: [{class: 'Beetroot', confidence: 0.90}, ...]}
Output: "Ingredient: Beetroot, Cuisine: Asian, Difficulty: beginner, Recipe:"
```

## 1. Environment Setup

In [ ]:
import json
from pathlib import Path
from typing import Dict, List, Optional

print("✅ Packages imported")

## 2. Selection Logic - Choose Primary Ingredient

In [ ]:
def select_primary_ingredient(roboflow_result: Dict) -> Optional[Dict]:
    """
    Select primary ingredient from multiple Roboflow detections
    
    Strategy: Choose highest confidence detection
    
    Args:
        roboflow_result: Roboflow inference result
        {
            'predictions': [
                {'class': 'Beetroot', 'confidence': 0.90, 'x': 100, 'y': 100, ...},
                {'class': 'Beetroot', 'confidence': 0.86, ...},
                ...
            ]
        }
    
    Returns:
        Primary ingredient detection or None
    """
    predictions = roboflow_result.get('predictions', [])
    
    if not predictions:
        return None
    
    # Select highest confidence
    primary = max(predictions, key=lambda p: p.get('confidence', 0))
    
    return primary

print("✅ Primary ingredient selection function defined")

## 3. Ingredient Name Normalization

In [ ]:
def normalize_ingredient_name(raw_name: str) -> str:
    """
    Normalize ingredient name for recipe generation
    
    Examples:
        'Beetroot' -> 'beetroot'
        'Chicken breast' -> 'chicken breast'
        'TOMATO' -> 'tomato'
    
    Args:
        raw_name: Raw class name from Roboflow
    
    Returns:
        Normalized ingredient name
    """
    # Convert to lowercase
    normalized = raw_name.lower()
    
    # Remove special characters (keep spaces)
    normalized = ''.join(c for c in normalized if c.isalnum() or c.isspace())
    
    # Strip whitespace
    normalized = normalized.strip()
    
    return normalized

# Test
print("✅ Name normalization function defined")
print(f"   Example: 'Beetroot' -> '{normalize_ingredient_name('Beetroot')}'")

## 4. Recipe Generation Prompt Builder

In [ ]:
def build_recipe_prompt(
    ingredient_name: str,
    cuisine_type: str = 'any',
    difficulty_level: str = 'beginner'
) -> str:
    """
    Build prompt for recipe generation model (GPT-2/Transformer)
    
    Format: "Ingredient: {name}, Cuisine: {type}, Difficulty: {level}, Recipe:"
    
    Args:
        ingredient_name: Normalized ingredient name
        cuisine_type: Cuisine type (Asian, Western, Fusion, any)
        difficulty_level: Difficulty (beginner, intermediate, advanced)
    
    Returns:
        Formatted prompt string
    """
    prompt = f"Ingredient: {ingredient_name}, Cuisine: {cuisine_type}, Difficulty: {difficulty_level}, Recipe:"
    return prompt

# Test
print("✅ Prompt builder function defined")
print(f"\n   Example prompt:")
print(f"   {build_recipe_prompt('beetroot', 'Asian', 'beginner')}")

## 5. Generate Multiple Diverse Prompts

In [ ]:
def generate_diverse_prompts(
    ingredient_name: str,
    num_recipes: int = 5
) -> List[Dict[str, str]]:
    """
    Generate diverse prompts for multiple recipes (per FR-004: min 3 cuisines)
    
    Strategy:
        - Recipe 1: Asian, beginner
        - Recipe 2: Western, beginner
        - Recipe 3: Fusion, intermediate
        - Recipe 4: any, beginner
        - Recipe 5: any, intermediate
    
    Args:
        ingredient_name: Ingredient name
        num_recipes: Number of recipes to generate (default 5)
    
    Returns:
        List of prompt configurations
    """
    # Define diverse combinations
    configurations = [
        {'cuisine': 'Asian', 'difficulty': 'beginner'},
        {'cuisine': 'Western', 'difficulty': 'beginner'},
        {'cuisine': 'Fusion', 'difficulty': 'intermediate'},
        {'cuisine': 'Mediterranean', 'difficulty': 'beginner'},
        {'cuisine': 'any', 'difficulty': 'intermediate'},
    ]
    
    # Generate prompts
    prompts = []
    for i in range(min(num_recipes, len(configurations))):
        config = configurations[i]
        prompt_text = build_recipe_prompt(
            ingredient_name,
            config['cuisine'],
            config['difficulty']
        )
        prompts.append({
            'prompt': prompt_text,
            'cuisine': config['cuisine'],
            'difficulty': config['difficulty'],
            'recipe_index': i + 1
        })
    
    return prompts

# Test
print("✅ Diverse prompts generator defined")
print(f"\n📊 Example: 5 diverse prompts for 'beetroot'")
test_prompts = generate_diverse_prompts('beetroot', 5)
for p in test_prompts:
    print(f"   {p['recipe_index']}. [{p['cuisine']}/{p['difficulty']}] {p['prompt']}")

## 6. Complete Adapter Function

In [ ]:
def adapt_detection_to_recipe_input(
    roboflow_result: Dict,
    num_recipes: int = 5
) -> Dict:
    """
    Complete adapter: Roboflow detection -> Recipe generation inputs
    
    Args:
        roboflow_result: Full Roboflow inference result
        num_recipes: Number of recipes to generate
    
    Returns:
        dict: {
            'success': bool,
            'ingredient': {
                'raw_name': str,
                'normalized_name': str,
                'confidence': float
            },
            'recipe_prompts': List[Dict],
            'error': str (if failed)
        }
    """
    # 1. Select primary ingredient
    primary = select_primary_ingredient(roboflow_result)
    
    if not primary:
        return {
            'success': False,
            'error': 'No ingredient detected'
        }
    
    # 2. Normalize ingredient name
    raw_name = primary.get('class', 'unknown')
    normalized_name = normalize_ingredient_name(raw_name)
    confidence = primary.get('confidence', 0)
    
    # 3. Generate diverse prompts
    prompts = generate_diverse_prompts(normalized_name, num_recipes)
    
    # 4. Return formatted result
    return {
        'success': True,
        'ingredient': {
            'raw_name': raw_name,
            'normalized_name': normalized_name,
            'confidence': confidence
        },
        'recipe_prompts': prompts,
        'num_prompts': len(prompts)
    }

print("✅ Complete adapter function defined")

## 7. Test with Example Roboflow Output

In [ ]:
# Simulate Roboflow output (based on your image)
example_roboflow_output = {
    'predictions': [
        {'class': 'Beetroot', 'confidence': 0.90, 'x': 200, 'y': 150, 'width': 120, 'height': 120},
        {'class': 'Beetroot', 'confidence': 0.87, 'x': 500, 'y': 300, 'width': 130, 'height': 130},
        {'class': 'Beetroot', 'confidence': 0.86, 'x': 400, 'y': 100, 'width': 125, 'height': 125},
        {'class': 'Beetroot', 'confidence': 0.80, 'x': 350, 'y': 200, 'width': 115, 'height': 115},
        {'class': 'Beetroot', 'confidence': 0.78, 'x': 450, 'y': 250, 'width': 110, 'height': 110},
    ]
}

# Run adapter
result = adapt_detection_to_recipe_input(example_roboflow_output, num_recipes=5)

# Display results
if result['success']:
    print("✅ Adapter Test Successful!\n")
    print(f"📊 Detected Ingredient:")
    print(f"   Raw name: {result['ingredient']['raw_name']}")
    print(f"   Normalized: {result['ingredient']['normalized_name']}")
    print(f"   Confidence: {result['ingredient']['confidence']:.2%}")
    
    print(f"\n🍳 Generated {result['num_prompts']} Recipe Prompts:")
    for prompt_config in result['recipe_prompts']:
        print(f"\n   Recipe {prompt_config['recipe_index']}:")
        print(f"   Cuisine: {prompt_config['cuisine']}")
        print(f"   Difficulty: {prompt_config['difficulty']}")
        print(f"   Prompt: \"{prompt_config['prompt']}\"")
else:
    print(f"❌ Adapter failed: {result['error']}")

## 8. Export Adapter Functions

In [ ]:
# Export functions for use in other notebooks
__all__ = [
    'select_primary_ingredient',
    'normalize_ingredient_name',
    'build_recipe_prompt',
    'generate_diverse_prompts',
    'adapt_detection_to_recipe_input'
]

print("✅ Adapter functions ready for export")
print(f"\n💡 Usage in recipe generation notebook:")
print("```python")
print("# Import adapter")
print("from adapter_to_recipe_generation import adapt_detection_to_recipe_input")
print("")
print("# Use it")
print("roboflow_result = CLIENT.infer('beetroot.jpg', model_id='food-ingredients-dataset/2')")
print("recipe_inputs = adapt_detection_to_recipe_input(roboflow_result, num_recipes=5)")
print("")
print("# Feed to recipe generation model")
print("for prompt_config in recipe_inputs['recipe_prompts']:")
print("    recipe = generate_recipe(prompt_config['prompt'])")
print("```")

## 9. Summary

### ✅ Adapter Functions Created:

1. **`select_primary_ingredient()`** - 從多個檢測中選最高置信度
2. **`normalize_ingredient_name()`** - 標準化食材名稱
3. **`build_recipe_prompt()`** - 構建單個提示詞
4. **`generate_diverse_prompts()`** - 生成5個多樣化提示詞
5. **`adapt_detection_to_recipe_input()`** - 完整轉換流程

### 🔄 Data Flow:

```
Roboflow Output                    Adapter                Recipe Generation Input
┌─────────────────┐               ┌────────┐              ┌──────────────────────┐
│ predictions: [  │               │        │              │ "Ingredient: beet-   │
│   {class: Beet- │──────────────>│ Select │─────────────>│  root, Cuisine:      │
│    root, conf:  │               │ Primary│              │  Asian, Difficulty:  │
│    0.90},       │               │   +    │              │  beginner, Recipe:"  │
│   {class: Beet- │               │ Normal-│              │                      │
│    root, conf:  │               │  ize   │              │ (× 5 diverse prompts)│
│    0.86},       │               │   +    │              │                      │
│   ...           │               │ Format │              │                      │
│ ]               │               │        │              │                      │
└─────────────────┘               └────────┘              └──────────────────────┘
```

### 📊 Output Format:

```json
{
  "success": true,
  "ingredient": {
    "raw_name": "Beetroot",
    "normalized_name": "beetroot",
    "confidence": 0.90
  },
  "recipe_prompts": [
    {
      "prompt": "Ingredient: beetroot, Cuisine: Asian, Difficulty: beginner, Recipe:",
      "cuisine": "Asian",
      "difficulty": "beginner",
      "recipe_index": 1
    },
    ...
  ],
  "num_prompts": 5
}
```

### 📝 Next Steps:

1. 整合到 `model_cnn_inference.ipynb`
2. 創建 T018-T022 的食譜生成 notebook
3. 使用這些 prompts 調用 GPT-2 生成食譜

In [ ]:
print("🎉 Roboflow → Recipe Generation Adapter ready!")
print("\n✅ Converts detection outputs to recipe generation inputs")
print("✅ Generates 5 diverse prompts (min 3 cuisines per FR-004)")
print("✅ Ready for integration with Transformer model")